# Stenosis Detection — Kaggle BUILD notebook (fast first run)

YOLO11n on **ARCADE task-2** (+ Danilov if attached). No nnU-Net teacher, so this is the quickest problem to a result — good to run on Kaggle **while coronary trains on Colab**.

### Kaggle setup (do this first)
1. **Settings → Accelerator → GPU** (T4 x2 or P100).
2. **Settings → Internet → ON** (needed for pip + git clone).
3. **Add Input → Datasets**: attach your **ARCADE** dataset (and Danilov if you have one). They appear under `/kaggle/input/<slug>/`.
4. Set `ARCADE_INPUT` below to the real path (run the `!ls /kaggle/input` cell to find it), then **Run All**.

All heavy logic is in the importable `src/*` package; this notebook just orchestrates it.

## 1 · GPU + code + install

In [ ]:
import os, sys, torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU.'
GITHUB_TOKEN = ''                                             # '' if repo PUBLIC, else paste a PAT
REPO_SLUG = 'jugalmodi0111/interventional-imaging-pipeline'
REPO = '/kaggle/working/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install 'ultralytics>=8.2' pycocotools opencv-python pyyaml
from src import env
E = env.setup()                                              # platform=kaggle, paths under /kaggle/working

## 2 · Point the config at your attached datasets
Kaggle mounts datasets read-only under `/kaggle/input/`. Find the exact folder, then symlink it into `data/raw/` so the configs resolve unchanged. `ARCADE_INPUT` must be the folder that **contains** `stenosis/` (and `syntax/`).

In [ ]:
import glob, os
# auto-discover the ARCADE root (folder that CONTAINS stenosis/ + syntax/), wherever Kaggle nested it
hits = glob.glob('/kaggle/input/**/stenosis/*/annotations/*.json', recursive=True)
assert hits, 'No ARCADE stenosis json under /kaggle/input — attach the ARCADE dataset (Add Input).'
ARCADE_INPUT = hits[0].split('/stenosis/')[0]     # parent of stenosis/
DANILOV_INPUT = ''                                 # leave '' — dca1 is a DIFFERENT (coronary) dataset, not Danilov
print('ARCADE_INPUT =', ARCADE_INPUT)
os.makedirs('data/raw', exist_ok=True)
!ln -sfn {ARCADE_INPUT} data/raw/arcade
if DANILOV_INPUT:
    !ln -sfn {DANILOV_INPUT} data/raw/danilov
n = len(glob.glob('data/raw/arcade/stenosis/**/*.json', recursive=True))
print('stenosis json via symlink:', n); assert n, 'symlink path wrong'

## 3 · Standardize ARCADE stenosis (+ Danilov) → YOLO

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/stenosis_yolo.yaml'))
if not DANILOV_INPUT:
    cfg['datasets'].pop('danilov', None)                     # ARCADE-only if no Danilov attached
from src.data_prep import danilov_to_yolo
danilov_to_yolo.main(cfg)
print('train imgs:', len(glob.glob('data/processed/stenosis/images/train/*')),
      '| val imgs:', len(glob.glob('data/processed/stenosis/images/val/*')))

## 4 · Train YOLO11n on GPU
Set `DRY_RUN = True` for a 3-epoch wiring check; `False` for the real 150-epoch run (target F1 > 0.57). Weights land in `/kaggle/working` (downloadable / savable as notebook output).

In [ ]:
DRY_RUN = True
if DRY_RUN:
    cfg['train']['epochs'] = 3
    cfg['ssl'] = {'pseudo_label': False}                     # skip SSL on the fast pass
from src.train.train_detector import train
best = train(cfg, project=f"{E['runs']}/stenosis")
print('best weights ->', best)

## 5 · Export handoff
Download `best.pt` from the notebook output. On your **Mac**: `python -m src.export.yolo_to_coreml --weights best.pt` → `.mlpackage` (one Ultralytics call, NMS baked in), then `src.serve.realtime --task det`.

In [ ]:
print('Download this file from the output panel, then export CoreML on the Mac:')
print(' ', best)